# Parkinson's Disease Detection Using Machine Learning

In [1]:
%pip install scikit-learn
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1. Project Introduction

Parkinson's disease is a progressive neurodegenerative disorder. Some changes associated with the disease can affect a person's voice.

The objective of this project is to use acoustic measurements to train a machine learning model that distinguishes recordings from people with Parkinson's disease from recordings from healthy people. The dataset comes from the UCI Machine Learning Repository.

This is an educational classification exercise, not a medical diagnostic tool.

## 2. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")

## 3. Loading the Dataset

In [ ]:
parkinsons_data = pd.read_csv("parkinson.csv.txt")

print("Dataset size:", parkinsons_data.shape)
parkinsons_data.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
parkinsons_data.info()

In [ ]:
parkinsons_data.describe()

In [ ]:
# Number of recordings in each class
parkinsons_data["status"].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=parkinsons_data, x="status")
plt.title("Distribution of the target variable")
plt.xlabel("Status (0 = healthy, 1 = Parkinson's)")
plt.ylabel("Number of recordings")
plt.show()

The dataset contains significantly more Parkinson’s recordings (147) than healthy recordings (48). This indicates a class imbalance, which may affect the model’s predictions. Therefore, accuracy alone may not be sufficient to evaluate its performance.

In [ ]:
parkinsons_data.groupby("status").mean(numeric_only=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=parkinsons_data, x="status", y="MDVP:Jitter(%)", ax=axes[0])
axes[0].set_title("Jitter according to status")
axes[0].set_xlabel("Status")
sns.boxplot(data=parkinsons_data, x="status", y="MDVP:Shimmer", ax=axes[1])
axes[1].set_title("Shimmer according to status")
axes[1].set_xlabel("Status")
plt.tight_layout()
plt.show()

Both Jitter and Shimmer show higher median values and greater variability for Parkinson’s recordings. This suggests that voice frequency and amplitude variations may be useful features for distinguishing between the two groups.


In [ ]:
# Correlations between the numerical variables
plt.figure(figsize=(12, 9))
sns.heatmap(parkinsons_data.drop(columns="name").corr(), cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

The matrix shows strong positive correlations between several Jitter-related features and between Shimmer-related features. Some voice measurements are also correlated with the target variable (status), suggesting that they may be useful for classification.

## 5. Data Preprocessing

The column `name` identifies the person and the recording. It is not used as a predictive variable. Since each person can have several recordings, recordings from the same person should not be split between the training and test sets. We therefore use the person identifier to create the split.

In [ ]:
X = parkinsons_data.drop(columns=["name", "status"])
Y = parkinsons_data["status"]

groups = parkinsons_data["name"].astype(str).str.rsplit("_", n=1).str[0]

# Split the data while keeping each participant in only one set
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_index, test_index = next(splitter.split(X, Y, groups=groups))

X_train = X.iloc[train_index]
X_test = X.iloc[test_index]
Y_train = Y.iloc[train_index]
Y_test = Y.iloc[test_index]

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nClass distribution in the test set:")
print(Y_test.value_counts())

print("\nParticipants in both sets:",
      len(set(groups.iloc[train_index]) & set(groups.iloc[test_index])))

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 6. Model Training

In [ ]:
# Train a Support Vector Machine (SVM) with a linear kernel
model = svm.SVC(kernel="linear")
model.fit(X_train, Y_train)

## 7. Model Evaluation

We evaluate the model on data that was not used during training. In addition to accuracy, precision, recall and F1-score provide a more complete view of the classification results, especially because the two classes are not equally represented.

In [ ]:
# Predictions on the training and test sets
Y_train_prediction = model.predict(X_train)
Y_test_prediction = model.predict(X_test)
print("Training accuracy:", accuracy_score(Y_train, Y_train_prediction))
print("Test accuracy:", accuracy_score(Y_test, Y_test_prediction))

In [ ]:
# Evaluation on the test set
print("Precision:", precision_score(Y_test, Y_test_prediction, zero_division=0))
print("Recall:", recall_score(Y_test, Y_test_prediction, zero_division=0))
print("F1-score:", f1_score(Y_test, Y_test_prediction, zero_division=0))

print("\nClassification report:")
print(classification_report(Y_test, Y_test_prediction, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(Y_test, Y_test_prediction)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Healthy", "Parkinson's"])
display.plot(cmap="Blues")
plt.title("Confusion matrix on the test set")
plt.show()

The model correctly classified 29 out of 37 recordings, achieving an accuracy of 78.4%. It correctly identified 26 Parkinson's recordings and 3 healthy recordings. However, 8 recordings were misclassified. The model performs better at identifying Parkinson's cases than healthy individuals, showing that its classification performance could still be improved.

## 8. Conclusion


This project explored the use of machine learning to distinguish between healthy individuals and people with Parkinson’s disease using voice measurements. The analysis revealed differences in vocal features, particularly Jitter and Shimmer, which may help identify Parkinson’s-related voice changes.

The model achieved an accuracy of 78.4% on the test set, correctly classifying 29 out of 37 recordings. Although these results are encouraging, some recordings were misclassified, especially those from healthy individuals. Further improvements, such as using a larger and more balanced dataset and testing different classification models, could help improve its performance.

## 9. Example Prediction

The following example uses one set of acoustic measurements to illustrate how the trained model makes a prediction. The output is a model classification only; it is not a medical conclusion.

In [ ]:
input_data = (197.07600, 206.89600, 192.05500, 0.00289, 0.00001,
              0.00166, 0.00168, 0.00498, 0.01098, 0.09700, 0.00563,
              0.00680, 0.00802, 0.01689, 0.00339, 26.77500, 0.422229,
              0.741367, -7.348300, 0.177551, 1.743867, 0.085569)

input_array = np.asarray(input_data).reshape(1, -1)
input_scaled = scaler.transform(input_array)
prediction = model.predict(input_scaled)

if prediction[0] == 0:
    print("Model prediction: healthy class (0)")
else:
    print("Model prediction: Parkinson's class (1)")